[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/13_svm_and_kernels/90_exercises.ipynb)

# Bài tập — Chương 13: SVM & Kernels (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/13_svm_and_kernels

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_classification_2d
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết bài toán hard-margin SVM, soft-margin SVM (có slack) và dạng hinge loss tương đương. Viết hàm dự đoán của kernel SVM theo các $\alpha_i$. Viết RBF kernel.

✍️ Trả lời:

**Đáp án:** Hard: $\min \frac12\lVert\mathbf w\rVert^2$ s.t. $y_i(\mathbf w^\top\mathbf x_i + b) \ge 1$. Soft: $\min \frac12\lVert\mathbf w\rVert^2 + C\sum\xi_i$ s.t. $y_i(\mathbf w^\top\mathbf x_i + b) \ge 1 - \xi_i$, $\xi_i \ge 0$ ⇔ $\min \frac12\lVert\mathbf w\rVert^2 + C\sum\max(0, 1 - y_i(\mathbf w^\top\mathbf x_i + b))$. Dự đoán: $f(\mathbf x) = \sum_i \alpha_i y_i K(\mathbf x_i, \mathbf x) + b$ (chỉ support vectors có $\alpha_i > 0$). RBF: $K(\mathbf a, \mathbf b) = \exp(-\gamma\lVert\mathbf a - \mathbf b\rVert^2)$.

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích "kernel trick" cho một người biết Logistic Regression nhưng chưa biết SVM: vì sao ta có thể làm việc trong không gian vô hạn chiều mà không tốn vô hạn tính toán?

✍️ Trả lời:

**Gợi ý:** để có ranh giới cong, ta có thể thêm feature phi tuyến (như feature mapping ở chương 07) — nhưng số feature bùng nổ. Điều kỳ diệu của SVM: khi viết bài toán ở dạng đối ngẫu, dữ liệu chỉ xuất hiện qua **tích vô hướng giữa các cặp điểm**. Nếu có một hàm $K(\mathbf a, \mathbf b)$ tính trực tiếp tích vô hướng **sau khi** biến đổi (mà không cần biến đổi thật), ta có toàn bộ lợi ích của không gian lớn với chi phí của một phép tính "độ giống nhau" giữa hai điểm. RBF kernel ứng với không gian vô hạn chiều nhưng chỉ cần tính $e^{-\gamma\lVert\mathbf a - \mathbf b\rVert^2}$.

## L2 — Tính tay

**Câu 3.** SVM tuyến tính có $\mathbf w = (3, 4)$, $b = -5$. (a) Độ rộng lề $2/\lVert\mathbf w\rVert$. (b) Với 3 điểm $\mathbf x_1 = (2, 1), y_1 = +1$; $\mathbf x_2 = (1, 0.5), y_2 = +1$; $\mathbf x_3 = (1, 1), y_3 = -1$: tính margin $m_i = y_i(\mathbf w^\top\mathbf x_i + b)$, hinge loss mỗi điểm và cho biết điểm nào là support vector / vi phạm lề / phân loại sai.

In [ ]:
w, b = np.array([3.0, 4.0]), -5.0
Xp = np.array([[2, 1], [1, 0.5], [1, 1.0]])
yp = np.array([1, 1, -1.0])
### START CODE HERE ### (≈ 3 dòng)
width = 2 / np.linalg.norm(w)
m = yp * (Xp @ w + b)
hinge = np.maximum(0, 1 - m)
### END CODE HERE ###
print("margin:", m, "| hinge:", hinge)

In [ ]:
check_close(width, 0.4, message="độ rộng lề")
check_close(m, [5, 0, -2], message="margin từng điểm")
check_close(hinge, [0, 1, 3], message="hinge loss")

✍️ Điểm nào phân loại sai? Điểm nào nằm **đúng trên** ranh giới?

**Đáp án:** $\mathbf x_1$: $m = 5 \ge 1$ — đúng, ngoài lề, hinge 0. $\mathbf x_2$: $m = 0$ — nằm đúng trên ranh giới, vi phạm lề (hinge 1). $\mathbf x_3$: $m = -2 < 0$ — **phân loại sai** (hinge 3). Điểm 2 và 3 có $m < 1$ → là support vectors trong soft-margin SVM.

**Câu 4.** Tính RBF kernel $K(\mathbf a, \mathbf b)$ với $\mathbf a = (1, 2)$, $\mathbf b = (2, 0)$, $\gamma = 0.5$; và polynomial kernel $(\mathbf a^\top\mathbf b + 1)^2$.

In [ ]:
a_vec, b_vec = np.array([1.0, 2.0]), np.array([2.0, 0.0])
### START CODE HERE ### (≈ 2 dòng)
k_rbf = np.exp(-0.5 * np.sum((a_vec - b_vec) ** 2))
k_poly = (a_vec @ b_vec + 1) ** 2
### END CODE HERE ###

In [ ]:
check_close(k_rbf, np.exp(-2.5), message="RBF: exp(−0.5·5)")
check_close(k_poly, 9, message="poly: (2 + 1)²")

## L3 — Code

**Bài 5.** Viết `hinge_objective_and_grad(w, b, X, y, C)` (nhãn ±1) trả về `(J, grad_w, grad_b)` với $J = \frac12\lVert\mathbf w\rVert^2 + C\sum\max(0, 1 - m_i)$ và subgradient (lấy 0 tại $m_i = 1$ — tức chỉ tính các điểm $m_i < 1$). Dùng nó viết `fit_linear_svm(X, y, C, lr, n_iters)` với learning rate $\frac{lr}{\sqrt{1 + t}}$.

In [ ]:
def hinge_objective_and_grad(w, b, X, y, C=1.0):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: m = None          # margin y·(Xw + b)
    # STARTER: viol = None       # mask m < 1
    # STARTER: J = None
    # STARTER: grad_w = None
    # STARTER: grad_b = None
    m = y * (X @ w + b)
    viol = m < 1
    J = 0.5 * w @ w + C * np.maximum(0, 1 - m).sum()
    grad_w = w - C * (y[viol, None] * X[viol]).sum(0)
    grad_b = -C * y[viol].sum()
    ### END CODE HERE ###
    return J, grad_w, grad_b


def fit_linear_svm(X, y, C=1.0, lr=0.005, n_iters=5000):
    w, b = np.zeros(X.shape[1]), 0.0
    for t in range(n_iters):
        ### START CODE HERE ### (≈ 3 dòng)
        # STARTER: # lấy subgradient, cập nhật w, b với bước lr / sqrt(1 + t)
        _, gw, gb = hinge_objective_and_grad(w, b, X, y, C)
        w = w - lr / np.sqrt(1 + t) * gw
        b = b - lr / np.sqrt(1 + t) * gb
        ### END CODE HERE ###
    return w, b

In [ ]:
Xl, yl = make_classification_2d("blobs", n=300, noise=0.6, seed=8)
Xl = StandardScaler().fit_transform(Xl)
yl_pm = np.where(yl == 1, 1.0, -1.0)
w_ours, b_ours = fit_linear_svm(Xl, yl_pm)
ref = LinearSVC(C=1.0, loss="hinge", max_iter=100_000, tol=1e-7).fit(Xl, yl_pm)
J_ours = hinge_objective_and_grad(w_ours, b_ours, Xl, yl_pm)[0]
J_ref = hinge_objective_and_grad(ref.coef_[0], ref.intercept_[0], Xl, yl_pm)[0]
print(f"objective: scratch = {J_ours:.4f} | LinearSVC = {J_ref:.4f}")
check_less((J_ours - J_ref) / J_ref, 0.01, "objective trong 1% so với LinearSVC")

**Bài 6.** Viết `rbf_kernel_matrix(A, B, gamma)` (không vòng lặp) và `kernel_decision_function(svc, X)` tái tạo `SVC.decision_function` từ `support_vectors_`, `dual_coef_` (= $\alpha_i y_i$), `intercept_` của một `SVC(kernel="rbf", gamma=<số>)` đã train.

In [ ]:
def rbf_kernel_matrix(A, B, gamma):
    ### START CODE HERE ### (≈ 2 dòng)
    sq = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T
    return np.exp(-gamma * np.maximum(sq, 0))
    ### END CODE HERE ###


def kernel_decision_function(svc, X):
    ### START CODE HERE ### (≈ 2 dòng)
    K = rbf_kernel_matrix(X, svc.support_vectors_, svc.gamma)        # gamma đã được đặt bằng một số
    return K @ svc.dual_coef_[0] + svc.intercept_[0]
    ### END CODE HERE ###

In [ ]:
Xk, yk = make_classification_2d("moons", n=300, noise=0.25, seed=2)
svc = SVC(kernel="rbf", C=1.0, gamma=2.0).fit(Xk, yk)
check_close(kernel_decision_function(svc, Xk[:50]), svc.decision_function(Xk[:50]), tol=1e-8,
            message="tái tạo decision_function từ support vectors")

## L4 — Debug

**Bài 7.** Pipeline sau cho accuracy kém dù đã "tune": (1) không scale (feature 2 có đơn vị lớn gấp 1000); (2) grid của `gamma` tuyến tính và quá lớn; (3) chọn tham số theo accuracy **train**. Sửa: Pipeline + `GridSearchCV` với `C` và `gamma` trên thang log (`np.logspace`), 5-fold. Lưu `best_model` và `test_acc`.

In [ ]:
Xd, yd = make_classification_2d("circles", n=600, noise=0.2, seed=4)
Xd = Xd * np.array([1.0, 1000.0])
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
### START CODE HERE ### (≈ 5 dòng)
# STARTER: best_acc, best_model = 0, None
# STARTER: for gamma in [1, 2, 3, 4, 5]:                                   # 🐛 tuyến tính, quá lớn
# STARTER:     m = SVC(kernel="rbf", gamma=gamma, C=100).fit(Xd_tr, yd_tr)    # 🐛 không scale
# STARTER:     if m.score(Xd_tr, yd_tr) > best_acc:                          # 🐛 accuracy train
# STARTER:         best_acc, best_model = m.score(Xd_tr, yd_tr), m
# STARTER: test_acc = best_model.score(Xd_te, yd_te)
search = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                      {"svc__C": np.logspace(-2, 3, 6), "svc__gamma": np.logspace(-3, 2, 6)}, cv=5).fit(Xd_tr, yd_tr)
best_model = search.best_estimator_
test_acc = best_model.score(Xd_te, yd_te)
### END CODE HERE ###
print(f"test accuracy = {test_acc:.3f}")

In [ ]:
check_greater(test_acc, 0.9, "test accuracy > 0.9")

## L5 — Phân tích

**Bài 8.** Với RBF SVM trên moons (n=300, noise=0.3), khảo sát số support vectors và accuracy validation theo `gamma` ∈ {0.01, 0.1, 1, 10, 100} (C = 1). Lưu vào DataFrame `res` (cột `gamma`, `n_sv`, `train_acc`, `val_acc`). Nhận xét quan hệ gamma ↔ overfit ↔ số support vectors.

In [ ]:
X8, y8 = make_classification_2d("moons", n=300, noise=0.3, seed=10)
X8v, y8v = make_classification_2d("moons", n=3000, noise=0.3, seed=11)
rows = []
### START CODE HERE ### (≈ 4 dòng)
# STARTER: # với mỗi gamma: SVC(kernel="rbf", C=1.0, gamma=gamma) → rows.append({"gamma", "n_sv", "train_acc", "val_acc"})
for gamma in [0.01, 0.1, 1, 10, 100]:
    m = SVC(kernel="rbf", C=1.0, gamma=gamma).fit(X8, y8)
    rows.append({"gamma": gamma, "n_sv": len(m.support_), "train_acc": m.score(X8, y8), "val_acc": m.score(X8v, y8v)})
### END CODE HERE ###
res = pd.DataFrame(rows)
res.round(3)

In [ ]:
check_greater(res.loc[res.gamma == 100, "train_acc"].item(), res.loc[res.gamma == 100, "val_acc"].item() + 0.05,
              "gamma = 100: train ≫ validation (overfit)")
check_greater(res["val_acc"].max(), res.loc[res.gamma == 0.01, "val_acc"].item(), "gamma quá nhỏ underfit")

✍️ Nhận xét:

**Gợi ý:** γ rất nhỏ → mỗi support vector ảnh hưởng rộng → ranh giới gần tuyến tính (underfit), rất nhiều SV vì nhiều điểm vi phạm lề. γ vừa phải → ranh giới theo đúng hình mặt trăng, val tốt nhất. γ rất lớn → mỗi SV chỉ ảnh hưởng vùng rất hẹp quanh nó → gần như mọi điểm train đều thành SV, train cao nhất nhưng validation tụt (overfit, giống KNN k nhỏ). Số SV lớn ở cả hai đầu nhưng vì lý do khác nhau.

## L6 — Thực tế

**Bài 9.** Phân loại chữ số viết tay (`load_digits_data`) bằng SVM: so sánh `LinearSVC`, `SVC(kernel="poly")`, `SVC(kernel="rbf")` (đã scale, tune `C` và `gamma`/`degree` bằng CV ≥ 3-fold). Báo cáo accuracy test của model tốt nhất (`best_test_acc`) — mục tiêu ≥ 0.98 — và in confusion matrix các cặp chữ số hay nhầm nhất.

In [ ]:
from sklearn.metrics import confusion_matrix

from ml_course.datasets import load_digits_data

Xg, yg, _ = load_digits_data()
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(Xg, yg, test_size=0.3, random_state=0, stratify=yg)
### START CODE HERE ### (≈ 10 dòng)
# STARTER: candidates = None     # dict tên → GridSearchCV(Pipeline(StandardScaler, model), lưới tham số, cv=3)
# STARTER: test_scores = None    # dict tên → accuracy test sau khi fit
# STARTER: best_name = None
# STARTER: best_test_acc = None
# STARTER: cm = None             # confusion matrix của model tốt nhất trên test
candidates = {
    "LinearSVC": GridSearchCV(make_pipeline(StandardScaler(), LinearSVC(max_iter=20_000)), {"linearsvc__C": [0.01, 0.1, 1]}, cv=3),
    "poly": GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="poly")), {"svc__C": [1, 10], "svc__degree": [2, 3]}, cv=3),
    "rbf": GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")), {"svc__C": [1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1]}, cv=3),
}
test_scores = {name: g.fit(Xg_tr, yg_tr).score(Xg_te, yg_te) for name, g in candidates.items()}
best_name = max(test_scores, key=test_scores.get)
best_test_acc = test_scores[best_name]
cm = confusion_matrix(yg_te, candidates[best_name].predict(Xg_te))
### END CODE HERE ###
print({k: round(v, 4) for k, v in test_scores.items()}, "→", best_name)
off = cm - np.diag(np.diag(cm))
pairs = sorted(((off[i, j], i, j) for i in range(10) for j in range(10) if off[i, j] > 0), reverse=True)[:3]
print("nhầm nhiều nhất (số lần, thật, dự đoán):", pairs)

In [ ]:
check_greater(best_test_acc, 0.98, "accuracy test ≥ 0.98")

## L7 — Nâng cao

**Bài 10.** **Random Fourier Features** (Rahimi & Recht, 2007): RBF kernel $K(\mathbf a, \mathbf b) = e^{-\gamma\lVert\mathbf a - \mathbf b\rVert^2}$ xấp xỉ bằng $z(\mathbf a)^\top z(\mathbf b)$ với $z(\mathbf x) = \sqrt{2/D}\cos(W^\top\mathbf x + \mathbf u)$, $W_{jk} \sim \mathcal N(0, 2\gamma)$, $u_k \sim U[0, 2\pi]$. Cài đặt `rff(X, D, gamma, seed)`; (a) chứng minh sai số xấp xỉ kernel giảm khi $D$ tăng; (b) train `LinearSVC` trên $z(\mathbf x)$ và so với `SVC(kernel="rbf")`.

In [ ]:
def rff(X, D=500, gamma=1.0, seed=0):
    r = np.random.default_rng(seed)
    ### START CODE HERE ### (≈ 3 dòng)
    W = r.normal(0, np.sqrt(2 * gamma), size=(X.shape[1], D))
    u = r.uniform(0, 2 * np.pi, D)
    Z = np.sqrt(2 / D) * np.cos(X @ W + u)
    ### END CODE HERE ###
    return Z

In [ ]:
Xf, yf = make_classification_2d("moons", n=2000, noise=0.25, seed=12)
Xf = StandardScaler().fit_transform(Xf)
K_true = rbf_kernel_matrix(Xf[:200], Xf[:200], gamma=1.0)
errors = {}
for D in [50, 500, 5000]:
    Z = rff(Xf[:200], D=D, gamma=1.0, seed=1)
    errors[D] = np.abs(Z @ Z.T - K_true).mean()
print("sai số trung bình |K̂ − K|:", {k: round(v, 4) for k, v in errors.items()})
check_less(errors[5000], errors[50] / 4, "sai số giảm khi D tăng (≈ 1/√D)")

Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.3, random_state=0)
Z_tr, Z_te = rff(Xf_tr, D=1000, gamma=1.0, seed=2), rff(Xf_te, D=1000, gamma=1.0, seed=2)   # cùng seed → cùng phép biến đổi
lin = LinearSVC(C=1.0, max_iter=20_000).fit(Z_tr, yf_tr)
acc_rff = lin.score(Z_te, yf_te)
acc_svc = SVC(kernel="rbf", gamma=1.0, C=1.0).fit(Xf_tr, yf_tr).score(Xf_te, yf_te)
print(f"accuracy test: RFF + LinearSVC = {acc_rff:.3f} | SVC(rbf) = {acc_svc:.3f}")
check_close(acc_rff, acc_svc, tol=0.02, message="RFF + model tuyến tính ≈ kernel SVM")